In [12]:
from pathlib import Path
from collections import defaultdict
from io import BytesIO
import re
import zipfile

import numpy as np
import pandas as pd


# ============================================================
# SETTINGS
# ============================================================
BASE = Path.cwd()
ZIP_NAME = "rpi_overhead.zip"

ARCHIVE = next(
    (
        path
        for path in (
            BASE / ZIP_NAME,
            Path("/mnt/data") / ZIP_NAME,
        )
        if path.exists()
    ),
    BASE / ZIP_NAME,
)

# Used only as a fallback if the ZIP is unavailable.
EXTRACTED_CSV = (
    BASE
    / "rpi_overhead_analysis"
    / "extracted_measurements.csv"
)

SCHEDULERS = ["DPS", "S²-EPSILON"]

FROM_LOG = {
    "EPSILON": "DPS",
    "SARTS": "S²-EPSILON",
}

METRICS = [
    ("selection", "Task-selection overhead (µs)"),
    ("randomization", "Arrival-randomization overhead (µs)"),
    ("performance", "Power: performance governor (W)"),
    ("ondemand", "Power: ondemand governor (W)"),
]


# ============================================================
# READ ARCHIVE FILES
# ============================================================
def read_zip_text(archive, suffix):
    matches = [
        name
        for name in archive.namelist()
        if name.endswith(suffix)
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Expected one ZIP entry ending in {suffix!r}, "
            f"found {len(matches)}: {matches}"
        )

    return archive.read(matches[0]).decode(errors="replace")


def read_zip_csv(archive, suffix):
    matches = [
        name
        for name in archive.namelist()
        if name.endswith(suffix)
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Expected one ZIP entry ending in {suffix!r}, "
            f"found {len(matches)}: {matches}"
        )

    return pd.read_csv(
        BytesIO(archive.read(matches[0]))
    )


# ============================================================
# TASK-SELECTION TIMING EXTRACTION
# ============================================================
def trace_pick_next_means(trace_text):
    """Extract completed pick_next_task_rt() durations in µs."""
    open_calls = defaultdict(list)
    durations = []

    pid_cpu = re.compile(r"-(\d+)\s+\[(\d{3})\]")
    exit_time = re.compile(
        r"funcgraph_exit:\s*([0-9.]+)\s*us"
    )

    for line in trace_text.splitlines():
        if (
            "pick_next_task_rt()" not in line
            and "funcgraph_exit:" not in line
        ):
            continue

        match = pid_cpu.search(line)

        if match is None or "|" not in line:
            continue

        key = (match.group(1), match.group(2))

        after_pipe = line.split("|", 1)[1]
        depth = len(after_pipe) - len(after_pipe.lstrip(" "))
        body = after_pipe.strip()

        if (
            "funcgraph_entry:" in line
            and body.startswith("pick_next_task_rt()")
            and body.endswith("{")
        ):
            open_calls[key].append(depth)

        elif (
            "funcgraph_exit:" in line
            and body == "}"
            and open_calls[key]
            and open_calls[key][-1] == depth
        ):
            match = exit_time.search(line)

            if match:
                durations.append(float(match.group(1)))
                open_calls[key].pop()

    return np.asarray(durations, dtype=float)


# ============================================================
# EXTRACT RUN/TRIAL MEANS
# ============================================================
def extract_measurements(archive_path):
    rows = []

    with zipfile.ZipFile(archive_path) as archive:
        for raw_scheduler, scheduler in FROM_LOG.items():
            for trial in range(1, 6):

                # Task-selection overhead: one mean per run.
                trace_suffix = (
                    "timing_repetitions/"
                    f"{raw_scheduler}_{trial}_trace.txt"
                )

                durations = trace_pick_next_means(
                    read_zip_text(archive, trace_suffix)
                )

                if len(durations) == 0:
                    raise ValueError(
                        "No completed pick_next_task_rt() calls "
                        f"found in {trace_suffix}"
                    )

                rows.append({
                    "metric": "selection",
                    "scheduler": scheduler,
                    "replicate": trial,
                    "value": float(durations.mean()),
                })

                # Arrival randomization: weighted mean per run.
                runtime_suffix = (
                    "timing_repetitions/"
                    f"{raw_scheduler}_{trial}_runtime.csv"
                )

                runtime = read_zip_csv(
                    archive,
                    runtime_suffix,
                )

                n_calls = int(
                    runtime["randomization_calls"].sum()
                )

                if n_calls <= 0:
                    raise ValueError(
                        "No randomization calls found in "
                        f"{runtime_suffix}"
                    )

                randomization_mean = float(
                    np.average(
                        runtime["mean_randomization_us"],
                        weights=runtime["randomization_calls"],
                    )
                )

                rows.append({
                    "metric": "randomization",
                    "scheduler": scheduler,
                    "replicate": trial,
                    "value": randomization_mean,
                })

        # Power: one mean per trial and governor.
        pmic = read_zip_csv(
            archive,
            "pmic_results/all_trials.csv",
        )

        for record in pmic.itertuples(index=False):
            if record.governor not in (
                "performance",
                "ondemand",
            ):
                continue

            if record.scheduler not in FROM_LOG:
                continue

            rows.append({
                "metric": record.governor,
                "scheduler": FROM_LOG[record.scheduler],
                "replicate": int(record.trial),
                "value": float(record.mean_pmic_rail_W),
            })

    return pd.DataFrame(rows)


# ============================================================
# VALIDATE AND SUMMARIZE
# ============================================================
def summarize_measurements(df):
    required_columns = {
        "metric",
        "scheduler",
        "replicate",
        "value",
    }

    missing = required_columns.difference(df.columns)

    if missing:
        raise ValueError(
            f"Missing required columns: {sorted(missing)}"
        )

    metric_order = [metric for metric, _ in METRICS]

    df = df[
        df["metric"].isin(metric_order)
        & df["scheduler"].isin(SCHEDULERS)
    ].copy()

    df["value"] = pd.to_numeric(
        df["value"],
        errors="raise",
    )

    for metric in metric_order:
        for scheduler in SCHEDULERS:
            group = df[
                (df["metric"] == metric)
                & (df["scheduler"] == scheduler)
            ]

            if (
                len(group) != 5
                or group["replicate"].nunique() != 5
            ):
                raise ValueError(
                    "Expected five run/trial means for "
                    f"{metric}, {scheduler}; "
                    f"found {len(group)} observations."
                )

            if not np.isfinite(group["value"]).all():
                raise ValueError(
                    f"Invalid values for {metric}, {scheduler}"
                )

    # Pandas std uses sample standard deviation: ddof=1.
    return (
        df.groupby(["metric", "scheduler"])["value"]
        .agg(["mean", "std"])
        .rename(columns={"std": "sd"})
    )


# ============================================================
# RUN — PRINT RESULTS ONLY
# ============================================================
if ARCHIVE.exists():
    measurements = extract_measurements(ARCHIVE)

elif EXTRACTED_CSV.exists():
    measurements = pd.read_csv(EXTRACTED_CSV)

else:
    raise FileNotFoundError(
        f"Place {ZIP_NAME} beside your notebook, "
        f"or provide {EXTRACTED_CSV}"
    )

summary = summarize_measurements(measurements)

rows = []

for metric, measurement in METRICS:
    row = {"Measurement": measurement}

    for scheduler, label in [
        ("DPS", "ε-Scheduler (mean ± SD)"),
        ("S²-EPSILON", "S²ε-Scheduler (mean ± SD)"),
    ]:
        result = summary.loc[(metric, scheduler)]

        row[label] = (
            f"{result['mean']:.3f} ± {result['sd']:.3f}"
        )

    rows.append(row)

paper_results = pd.DataFrame(rows)

print(paper_results.to_string(index=False))

                        Measurement ε-Scheduler (mean ± SD) S²ε-Scheduler (mean ± SD)
       Task-selection overhead (µs)           1.074 ± 0.009             1.090 ± 0.048
Arrival-randomization overhead (µs)           0.210 ± 0.007             0.169 ± 0.007
    Power: performance governor (W)           3.623 ± 0.083             3.610 ± 0.041
       Power: ondemand governor (W)           2.818 ± 0.076             2.857 ± 0.089
